# ReAct Pattern - Thought, Action, Observation


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2fd02b2eaa-16c3-4f92-8f97-06329fbcccd4_716x550-7.gif" alt="ReAct pattern" width="500"/>

ReAct is a bounded workflow for answering a question through repeated **thought -> action -> observation -> response** cycles.
In this lab, we will use a small forensic mini-case and three simple tools to answer one communication-timing question without guessing.
In Lab 3, the `Environment` is the staged forensic evidence package, which the agent accesses indirectly through tools that read artifact files such as `incident_window.csv`, `messaging_events.csv`, and `network_events.csv`.

Your job in this notebook is to restate the forensic question, run the ReAct loop one tool call at a time, inspect each observation before moving to the next step, and then compare your manual process with the packaged `ReactAgent`.

This is the **third pattern lab** in the course. If you want to revisit the earlier lessons first:

* [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
* [Tool Use Pattern](../lab2_tool_use_pattern/03_lab_notebook.ipynb)


## Relevant Imports and LLM Client


Run this setup cell first. It loads the lab configuration from `.env`, checks that you opened the notebook from the correct folder, connects to the model client, and points the notebook to the Lab 3 data files.

You do not need to understand every Python line here. If this cell fails, fix the setup issue before moving on.


In [ ]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab3_react_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 3 data folder')

from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


## Define Forensic Tools

These tools expose one narrow part of the mini-case at a time so the ReAct loop has to inspect the evidence step by step.


This cell defines the three forensic tools used in the lab. Each tool reveals one small part of the case so the model has to gather evidence step by step instead of guessing.

Tools introduced here, with simple examples:

- `get_incident_window()`
  Example output: `{"start": "2026-02-20T14:10:00Z", "end": "2026-02-20T14:25:00Z", "start_source": "staff_observation", "end_source": "staff_observation", "start_note": "phone left on supply table before coordinator stepped away", "end_note": "phone recovered from supply table when coordinator returned"}`

- `get_message_attempt(app="Signal")`
  Example output: `{"timestamp_utc": "2026-02-20T14:16:11Z", "app": "Signal", "event": "attachment_attempt", "contact": "+1-555-0179", "details": "outgoing image attachment attempt recorded while mobile data unavailable"}`

- `get_network_restore_time()`
  Example output: `"2026-02-20T14:28:02Z"`


In [ ]:
def read_csv(filename: str) -> list[dict]:
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


@tool
def get_incident_window() -> str:
    """Return the start and end timestamps for the reported unattended interval."""
    rows = read_csv('incident_window.csv')
    start = next(row for row in rows if row['event'] == 'unattended_interval_start')
    end = next(row for row in rows if row['event'] == 'unattended_interval_end')
    return json.dumps(
        {
            'start': start['timestamp_utc'],
            'end': end['timestamp_utc'],
            'start_source': start['source'],
            'end_source': end['source'],
            'start_note': start['notes'],
            'end_note': end['notes'],
        }
    )


@tool
def get_message_attempt(app: str) -> str:
    """Return the first recorded messaging attachment attempt for the requested app."""
    rows = read_csv('messaging_events.csv')
    match = next((row for row in rows if row['app'].lower() == app.lower()), None)
    if match is None:
        return json.dumps({'error': f'No messaging event found for {app}.'})
    return json.dumps(match)


@tool
def get_network_restore_time() -> str:
    """Return the timestamp when mobile data connectivity was restored."""
    rows = read_csv('network_events.csv')
    restored = next(row for row in rows if row['status'] == 'restored')
    return restored['timestamp_utc']


available_tools = {
    'get_incident_window': get_incident_window,
    'get_message_attempt': get_message_attempt,
    'get_network_restore_time': get_network_restore_time,
}


This quick check prints the name and input signature for one tool.

Why this matters: it shows that an ordinary Python function has been wrapped into a tool the model can call in a structured way.


In [ ]:
print('Tool name:', get_incident_window.name)
print('Tool signature:', get_incident_window.fn_signature)


## A System Prompt for the ReAct Loop


This cell defines the system prompt that teaches the model how to behave in a ReAct loop.

Pay attention to the four required stages: `thought`, `action`, `observation`, and `response`. The prompt also restricts the model to one tool call at a time.


In [ ]:
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of four stages:

1. Thought — reason about what you need next. Wrap in <thought></thought> tags.
2. Action — call one tool in <tool_call></tool_call> tags.
3. Observation — the tool result will be returned in <observation></observation> tags.
4. Response — when you have enough evidence, answer in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

## Available Tools
<tools>
%s
</tools>

## Additional Constraints
- Use exactly one tool call at a time.
- Do not skip directly to a final answer when evidence is still missing.
- Keep the final answer evidence-bounded.
"""


## Part 1: Walk the ReAct Loop Manually

For the manual walkthrough, we will use a prompt that forces a predictable tool order so you can inspect each turn clearly.
As you run each step, pay attention to four things: the question being answered, the tool call chosen, the observation returned, and why that observation changes the next step.
Do not jump to the final answer until the needed observations have been collected.


This cell prepares the manual walkthrough. It builds the tool list, states the forensic question, starts the conversation history, and defines a helper function that runs one ReAct turn.

This is the core mechanism of the lab: ask a question, make one tool call, inspect the observation, then decide what should happen next.


In [ ]:
tools_signature = ',\n'.join(
    [
        get_incident_window.fn_signature,
        get_message_attempt.fn_signature,
        get_network_restore_time.fn_signature,
    ]
)
formatted_system_prompt = REACT_SYSTEM_PROMPT % tools_signature

USER_QUESTION = (
    'Determine whether the Signal attachment attempt happened during the reported unattended interval ' 
    'and whether the device reconnected before the interval ended. ' 
    'Use exactly one tool call per turn in this order: get_incident_window, ' 
    'then get_message_attempt for Signal, then get_network_restore_time. ' 
    'After the third observation, provide a final answer.'
)

chat_history = [
    {'role': 'system', 'content': formatted_system_prompt},
    {'role': 'user', 'content': f'<question>{USER_QUESTION}</question>'},
]


def run_single_react_turn(history: list[dict]):
    output = client.chat.completions.create(messages=history, model=MODEL).choices[0].message.content
    print(output)
    history.append({'role': 'assistant', 'content': output})

    tool_call = extract_tag_content(output, tag='tool_call')
    if not tool_call.found:
        return output, None, None

    tool_call_dict = json.loads(tool_call.content[0])
    tool_result = available_tools[tool_call_dict['name']].run(**tool_call_dict['arguments'])
    print('\nTool call:', tool_call_dict)
    print('Tool result:', tool_result)
    history.append({'role': 'user', 'content': f'<observation>{tool_result}</observation>'})
    return output, tool_call_dict, tool_result


### Step 1


This is Step 1 of the manual ReAct loop. Run it and inspect what the model chooses to do first.

What to look for: the model should identify that it first needs the unattended interval before it can judge whether the Signal event happened inside or outside that window.


In [ ]:
output_1, tool_call_1, tool_result_1 = run_single_react_turn(chat_history)


### Step 2


This is Step 2 of the manual ReAct loop. Run it after reviewing the first observation.

What to look for: the next tool call should focus on the Signal attachment attempt, because the model now knows the time window and can compare the message timestamp against it.


In [ ]:
output_2, tool_call_2, tool_result_2 = run_single_react_turn(chat_history)


### Step 3


This is Step 3 of the manual ReAct loop. Run it after reviewing the message observation.

What to look for: the model should now check when connectivity returned so it can answer the second part of the forensic question.


In [ ]:
output_3, tool_call_3, tool_result_3 = run_single_react_turn(chat_history)


### Final Response


This cell asks the model for a final response after the needed observations have been collected.

A good final answer should cite the observed timestamps, answer both parts of the question, and avoid claiming more than the evidence supports.


In [ ]:
final_output = client.chat.completions.create(messages=chat_history, model=MODEL).choices[0].message.content
print(final_output)


## Part 2: Run the Same Workflow with `ReactAgent`

The local `ReactAgent` wraps the same loop for you, so you can focus on the forensic question rather than the message bookkeeping.


This cell creates the packaged `ReactAgent` using the same three tools from the manual walkthrough.

Why this matters: students can compare the explicit step-by-step process they just inspected with a wrapped version that automates the same pattern.


In [ ]:
from agentic_patterns.react_pattern.react_agent import ReactAgent

react_agent = ReactAgent(
    tools=[get_incident_window, get_message_attempt, get_network_restore_time],
    client=client,
    model=MODEL,
)


This cell runs the packaged `ReactAgent` on the same forensic question.

Compare its behavior to your manual walkthrough: what tool did it choose first, what observations did it rely on, and did its final answer stay evidence-bounded?


In [ ]:
react_agent.run(user_msg=USER_QUESTION)
